# Sizing

So far every `size` and `capacity` has been a number we picked by hand. **`Sizing`** lets the optimizer pick them, trading the cost of building bigger against the cost of operating smaller.

You'll meet **`Sizing`** — a parameter object that turns a fixed `size` (or `capacity`) into a decision variable, optionally with per-MW investment cost.

In [ ]:
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from fluxopt import Carrier, Converter, Effect, Flow, Port, Sizing, Storage, optimize

pio.renderers.default = 'notebook_connected'

## 1. From numbers to `Sizing`

Anywhere you wrote a fixed `size=120` or `capacity=300`, you can write `Sizing(min_size=..., max_size=..., effects_per_size=...)` instead. The fields:

- `min_size` / `max_size` — the bracket the optimizer searches.
- `effects_per_size` — investment cost per unit, charged once. `{'cost': 0.27}` means €0.27 per kW of installed capacity.
- `mandatory=True` (default) — must build something within `[min, max]`. Set `False` to let the optimizer choose not to build at all (introduces a binary).

We use **daily-amortized** rates here so that the 24-hour run is meaningful: €0.27/kW/day for the boiler (~€1000/kW lifetime over 10 years), €0.027/kWh/day for the tank. T5 will show how multi-period models handle proper investment timing across years.

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]
gas_price = [0.04 if h < 6 or h >= 22 else 0.12 for h in range(n)]
demand = [
    10,
    10,
    8,
    8,
    8,
    12,
    25,
    60,
    70,
    75,
    75,
    70,
    65,
    60,
    55,
    50,
    45,
    40,
    30,
    25,
    20,
    15,
    12,
    10,
]
peak = max(demand)

result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    ports=[
        Port(
            'gas_grid',
            imports=[
                Flow('gas', size=1000, effects_per_flow_hour={'cost': gas_price}),
            ],
        ),
        Port(
            'workshop',
            exports=[
                Flow('heat', size=peak, fixed_relative_profile=[d / peak for d in demand]),
            ],
        ),
    ],
    converters=[
        Converter.boiler(
            'boiler',
            thermal_efficiency=0.9,
            fuel_flow=Flow('gas', size=300),
            thermal_flow=Flow(
                'heat',
                size=Sizing(min_size=20, max_size=200, effects_per_size={'cost': 0.27}),
            ),
        ),
    ],
    storages=[
        Storage(
            'tank',
            capacity=Sizing(min_size=0, max_size=1000, effects_per_size={'cost': 0.027}),
            charging=Flow('heat', size=80),
            discharging=Flow('heat', size=80),
            eta_charge=0.98,
            eta_discharge=0.98,
            relative_loss_per_hour=0.005,
        ),
    ],
    objective_effects='cost',
)
print(f'Total cost (operating + investment): {result.objective:.2f} EUR')

## 2. The chosen sizes

`result.sizes` is a `flow`-indexed array (NaN for fixed-size flows); `result.storage_capacities` is `storage`-indexed. Drop the NaNs to get just the optimized values.

In [ ]:
result.sizes.dropna(dim='flow').to_dataframe(name='size [kW]')

In [ ]:
result.storage_capacities.to_dataframe(name='capacity [kWh]')

In [ ]:
times = result.flow_rates.coords['time'].values
level = result.storage_level('tank')

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=('Flow rates', 'Tank level')
)
for fid in result.flow_rates.coords['flow'].values:
    fig.add_trace(
        go.Scatter(x=times, y=result.flow_rate(str(fid)).values, name=str(fid), line_shape='hv'),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(
        x=level.coords['time'].values, y=level.values, name='level', fill='tozeroy', line_shape='hv', showlegend=False
    ),
    row=2,
    col=1,
)
fig.update_layout(height=420, margin={'l': 50, 'r': 20, 't': 30, 'b': 20}, template='plotly_white')
fig.update_yaxes(title_text='kW', row=1, col=1)
fig.update_yaxes(title_text='kWh', row=2, col=1)
fig

The optimizer picks a boiler smaller than the demand peak (75 kW) — it's cheaper to build less boiler and arbitrage across the TOU spread with a generous tank. Push the tank's `effects_per_size` up and the answer flips: the boiler grows toward the peak and the tank shrinks.

## Recap

`Sizing` turns design choices into LP variables. Pair it with `effects_per_size` to capture investment cost; pair `mandatory=False` with `min_size > 0` to give the optimizer a yes/no build choice (introduces a binary).

**Next**: [Multi-period & Investment](05-multi-period.ipynb) — handle multiple periods (years) at once and decide *when* to build, not just *whether*.